# 🚀 NairaLLM V1.5 — Free Cloud GPU Semantic Pretraining Pilot

**Objective**: Run the first real free-cloud GPU semantic pretraining pilot on Google Colab (Tesla T4 GPU).

### Verification Highlights:
- **Dataset**: Dataset A (`semantic_pretrain_v1_5_expanded.jsonl` — 337 records, 105,141 BPE tokens, clean provenance, 0 duplicates, 20 domains)
- **Hardware**: Google Colab Free Tier (Tesla T4 GPU, ~14.56 GB VRAM)
- **Cost**: **$0.00** (Zero paid compute / No paid compute units used)
- **Scope**: Short Pilot Run + 11-step Preflight + 7-domain Semantic Benchmark + Google Drive Checkpointing + STOP Gate.

## 🛠️ Step 1: Environment & Free GPU Hardware Diagnostics

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM:      {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"AMP Supported:   True")
else:
    print("WARNING: GPU runtime not active. Please select Runtime -> Change runtime type -> T4 GPU.")

## 📂 Step 2: Mount Google Drive for Persistent Checkpointing

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')
colab_ckpt_dir = '/content/drive/MyDrive/Naira-Training/checkpoints/semantic_pretrain_pilot'
os.makedirs(colab_ckpt_dir, exist_ok=True)
print(f"Persistent Google Drive Checkpoint Directory Ready: {colab_ckpt_dir}")

## 📦 Step 3: Setup NairaLLM Workspace & Dependencies

In [ ]:
# If running directly from cloned repo:
# %cd /content/naira-os

!pip install -q tokenizers psutil

import sys
from pathlib import Path
workspace_root = Path('.').resolve()
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

print("Workspace initialized successfully.")

## ⚡ Step 4: Run Semantic Pretraining Pilot (Phases 1 — 5)

In [ ]:
from NairaLLM.training.scripts.run_semantic_pilot import run_semantic_pilot

# Execute the short pilot on Dataset A (Tesla T4 GPU with AMP mixed precision)
results = run_semantic_pilot(
    epochs=10,
    batch_size=4,
    grad_accum_steps=4,
    learning_rate=4e-4,
    max_seq_len=256,
    custom_checkpoint_dir=colab_ckpt_dir,
)

print(f"\n[GATE] Pilot Finished with Recommendation: {results.get('recommendation')}")

## 📊 Step 5: Display Markdown Report & Google Drive Sync Status

In [ ]:
from IPython.display import display, Markdown

report_path = 'NairaLLM/evaluation/results/semantic_pretraining_pilot.md'
if os.path.exists(report_path):
    with open(report_path, 'r', encoding='utf-8') as f:
        content = f.read()
    display(Markdown(content))

print("\nPersistent Checkpoints in Google Drive:")
!ls -lh "/content/drive/MyDrive/Naira-Training/checkpoints/semantic_pretrain_pilot/"